In [ ]:
path_prefix = '../..'
import sys
sys.path.append(path_prefix)

import logging
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from egh_vlm.hallu_dataset import load_hallu_dataset, split_stratified, hallu_collate_fn
from egh_vlm.hallu_ffn_detector import HalluFFNDetector, train_ffn_detector, eval_ffn_detector
from egh_vlm.utils import ModelBundle

### Analysis

In [ ]:
feature_folder = f'{path_prefix}/data/phd/feature_diff_layers'
dataset_path = f'{path_prefix}/data/phd/phd_base_sample.json'
img_folder_path  = f'{path_prefix}/data/phd/images'
model_name = 'Qwen/Qwen3-VL-2B-Instruct'
train_ratio = 0.7

In [ ]:
features = {}

for feature_name in sorted(os.listdir(feature_folder)):
    if not feature_name.endswith('.pt') or not feature_name.startswith('features_layer_'):
        continue

    layer = int(feature_name[len('features_layer_'):-3])
    feature_path = os.path.join(feature_folder, feature_name)
    features[layer] = load_hallu_dataset(feature_path)

print(f'Loaded feature layers: {sorted(features)}')
hidden_size = features[8].embs[0].size(-1) if len(features) > 0 else 0
print(f'Hidden size of features: {hidden_size}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device used: {device}')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    dtype='auto',
    device_map=device
)
processor = AutoProcessor.from_pretrained(
    model_name,
    max_pixels=1024 * 768)

model_bundle = ModelBundle(model, processor, device)

In [ ]:
def training_pipeline(train_dataloader, val_dataloader, weight, epoch, lr, save_path='log.txt'):
    logging.basicConfig(filename=save_path, level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')

    result = {
        'w': weight,
        'train_ratio': train_ratio,
        'epoch': epoch,
        'lr': lr,
        'acc': {
            'value': 0.0,
            'epoch': -1
        },
        'f1': {
            'value': 0.0,
            'epoch': -1
        },
        'pr_auc': {
            'value': 0.0,
            'epoch': -1
        }
    }

    # Init
    detector = HalluFFNDetector(hidden_size, hidden_size, 1, weight)
    loss_fn = nn.BCELoss()
    optim = torch.optim.Adam(detector.parameters(), lr=lr)
    
    logging.debug(f'Training w/ weight: {weight}')
    for i in range(epoch):
        total_loss = train_ffn_detector(detector, loss_fn, optim, train_dataloader)
        eval_result = eval_ffn_detector(detector, val_dataloader)
        acc, f1, pr_auc =  eval_result['acc'], eval_result['f1'], eval_result['pr_auc']

        logging.debug(f'Epoch [{i+1}/{epoch}], Loss: {total_loss:.4f}')
        logging.debug(f'Epoch [{i + 1}/{epoch}], ACC: {acc:.4f}, F1: {f1:.4f}, PR-AUC:{pr_auc:.4f}\n')
        if acc > result['acc']['value']:
            result['acc']['value'] = acc
            result['acc']['epoch'] = i + 1
        if f1 > result['f1']['value']:
            result['f1']['value'] = f1
            result['f1']['epoch'] = i + 1
        if pr_auc > result['pr_auc']['value']:
            result['pr_auc']['value'] = pr_auc
            result['pr_auc']['epoch'] = i + 1
        if total_loss < 1e-4:
            break
    logging.debug(f'Eval ACC: {result["acc"]["value"]:.4f} at epoch {result["acc"]["epoch"]}')
    logging.debug(f'Eval F1: {result["f1"]["value"]:.4f} at epoch {result["f1"]["epoch"]}')
    logging.debug(f'Eval PR-AUC: {result["pr_auc"]["value"]:.4f} at epoch {result["pr_auc"]["epoch"]}')

    # Clean up logger handlers
    logger = logging.getLogger()
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

    return result

In [ ]:
weight = 0.7
epoch = 30
lr = 1e-4
records = []

for layer in features:
    feature = features[layer]

    train_dataset, val_dataset = split_stratified(feature, train_ratio=train_ratio, random_state=42)
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=16,
        collate_fn=hallu_collate_fn,
        shuffle=True,
    )
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=16,
        collate_fn=hallu_collate_fn,
        shuffle=True,
    )
    result = training_pipeline(train_dataloader, val_dataloader, weight, epoch, lr, save_path=f'{path_prefix}/data/phd/logs/egh_vlm_sample_{layer}_log.txt')
    result['layer'] = layer
    records.append(result)